# Module 1 — KNN: Is This a Peak Hour?
**Lectures:** Week 2 (fit → predict → accuracy) · Week 3 (functions, scatter plots, KNN, Euclidean distance)

**Real-world use:** large customers (factories, data centres) pay "Global Adjustment" charges based on their use during Ontario's top peak hours (ICI program). Flagging peak hours = real money saved.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ieso_lab import load_dataset
from ieso_lab.features import add_peak_label
from ieso_lab.tracking import log_run, show_runs

plt.rcParams["figure.figsize"] = (9, 4)
df = load_dataset().dropna().reset_index(drop=True)
print(df.shape)

## 1. Create the class label
Peak = demand in the top 10%. Threshold from the **training set only** (Module 2 explains why).

In [ ]:
from sklearn.model_selection import train_test_split
FEATURES = ["hour", "temperature_2m"]
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
threshold = train_df["demand_mw"].quantile(0.90)
train_df = add_peak_label(train_df.copy(), threshold)
test_df = add_peak_label(test_df.copy(), threshold)
X_train, y_train = train_df[FEATURES], train_df["is_peak"]
X_test, y_test = test_df[FEATURES], test_df["is_peak"]
print(f"Peak threshold: {threshold:,.0f} MW | peak share: {y_train.mean():.1%}")

## 2. Scatter plot (Week 3) — can you *see* the boundary?

In [ ]:
colors = ["tab:blue", "tab:red"]
plt.scatter(X_train["hour"], X_train["temperature_2m"], c=[colors[i] for i in y_train], s=3, alpha=0.4)
plt.xlabel("Hour (EST)"); plt.ylabel("Temperature (°C)"); plt.title("Red = peak hour")

## 3. Euclidean distance by hand
$d(x,z)=\sqrt{\sum_i (x_i-z_i)^2}$. KNN predicts the majority class among the *k* smallest distances.

In [ ]:
new_hour = np.array([17, 31.0])   # 5 PM, 31°C
# TODO 3: compute the distance from new_hour to EVERY row of X_train (numpy, no loops)
dists = None
# TODO 4: indices of the 5 nearest rows, then their labels
nearest_idx = None
print("labels of 5 nearest:", y_train.iloc[nearest_idx].values)

## 4. Train the KNN classifier (Import → Create → Fit → Predict → Evaluate)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
# TODO 5: create a KNN with k=3, fit on train, predict on test, compute accuracy
model = None
y_pred = None
acc = None
print("Accuracy:", acc)

In [ ]:
log_run("01", "knn", {"k": 3, "features": FEATURES}, {"accuracy": acc})

## 5. 🔨 Break it on purpose: what does *k* do?
**Predict first:** with k=1, will training accuracy be high or low? Test accuracy?

In [ ]:
ks = [1, 3, 5, 15, 51, 151, 501]
tr, te = [], []
for k in ks:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    tr.append(m.score(X_train, y_train)); te.append(m.score(X_test, y_test))
plt.plot(ks, tr, "o-", label="train"); plt.plot(ks, te, "o-", label="test")
plt.xscale("log"); plt.xlabel("k"); plt.ylabel("accuracy"); plt.legend()

## 6. 🔨 The accuracy trap (sets up Week 5)
A "model" that **always says not-peak** — what accuracy does it get?

In [ ]:
print("Always-0 accuracy:", (y_test == 0).mean())

## 7. Why questions
1. Why does k=1 score ~100% on training data? (Hint: what is each point's nearest neighbour?)
2. Your KNN accuracy vs the always-0 baseline — is your model actually useful? What metric would tell you better?
3. Is KNN "learning a function" or "memorising"? What does `fit()` actually store?

---
## 🔧 Git task (do this before you start, and finish after)
```bash
git switch main && git pull
git switch -c module-01-knn          # start of module
# ... work ...
git add notebooks/ experiments/runs.csv
git commit -m "Module 1: KNN peak classifier + k sweep"
git push -u origin module-01-knn
```
Then open a **Pull Request** on GitHub → read your own diff → merge → close the module's Issue.
